# Semantic Causal Pathway Consistency in MLLMs

This notebook implements the staged architecture in `Semantic_Causal_Pathway_Consistency_MLLM_Architecture.md`: question-critical evidence, semantically validated variants, external necessity/sufficiency, Qwen2.5-VL answer likelihoods, prompt-side attention mediator scans, activation patching, mediator recovery, CPS, CECA, and SCC.

Run top to bottom. Data and model downloads are opt-in. Do not mark a variant valid without inspecting it. Heavyweight causal cells require a GPU, a fixed answer candidate set, a critical evidence mask, and at least two manually accepted variants. The generated demo is only a plumbing example, never benchmark data.

## 1. Environment and portable paths

In [ ]:
from pathlib import Path
import json, sys, platform, time, random, importlib.metadata as package_metadata
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir(): raise RuntimeError('Open from the repository or notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Python:', platform.python_version(), '|', platform.platform())
print('Project:', PROJECT_ROOT)

## 2. Experiment configuration

Set data paths for this PC in `configs/experiment.json`. Absolute Windows drive paths and portable relative paths are both accepted. Keep datasets, HF cache, and checkpoints outside the repository.

In [ ]:
config = json.loads((PROJECT_ROOT / 'configs' / 'experiment.json').read_text(encoding='utf-8'))
random.seed(int(config.get('seed',17))); np.random.seed(int(config.get('seed',17)))
DATA_ROOT = (PROJECT_ROOT / config['data_root']).resolve()
MANIFEST = (PROJECT_ROOT / config['manifest']).resolve()
OUTPUT_DIR = (PROJECT_ROOT / config['output_dir']).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({**config, 'data_root_resolved':str(DATA_ROOT), 'manifest_resolved':str(MANIFEST), 'output_resolved':str(OUTPUT_DIR)}, indent=2))

## 3. Load manifest or create a smoke-run sample

Manifest schema and portability notes are in `data/README.md`. The demo image/mask below exists only to validate the lightweight cells when you have not configured a benchmark yet.

In [ ]:
def load_jsonl(path):
    rows=[]
    with Path(path).open(encoding='utf-8-sig') as f:
        for line_no,line in enumerate(f,1):
            if line.strip():
                row=json.loads(line)
                if not row.get('image') or not row.get('question'): raise ValueError(f'Line {line_no} requires image and question')
                rows.append(row)
    return rows

def resolve_path(value, data_root, manifest=None):
    p=Path(value).expanduser()
    if p.is_absolute(): return p
    for candidate in [Path(data_root)/p, Path(manifest).parent/p if manifest else Path(data_root)/p]:
        if candidate.exists(): return candidate.resolve()
    return (Path(data_root)/p).resolve()

if MANIFEST.exists():
    rows=load_jsonl(MANIFEST)[:int(config.get('max_samples',1))]
    for row in rows: row['image_path']=resolve_path(row['image'],DATA_ROOT,MANIFEST)
    missing=[str(r['image_path']) for r in rows if not r['image_path'].is_file()]
    if missing: raise FileNotFoundError('Missing image paths, e.g. '+missing[0])
    active=rows[0]
    IS_DEMO=False
else:
    demo_dir=OUTPUT_DIR/'demo_data'; demo_dir.mkdir(parents=True,exist_ok=True)
    demo_image=Image.new('RGB',(384,256),'#e8e8e8'); draw=ImageDraw.Draw(demo_image)
    draw.ellipse((135,45,250,160),fill='#d62828',outline='black',width=4); draw.polygon([(135,100),(250,100),(285,130),(100,130)],fill='#d62828',outline='black')
    image_path=demo_dir/'demo_umbrella.png'; mask_path=demo_dir/'demo_evidence_mask.png'
    demo_image.save(image_path)
    mask=Image.new('L',demo_image.size,0); md=ImageDraw.Draw(mask); md.ellipse((135,45,250,160),fill=255); md.polygon([(135,100),(250,100),(285,130),(100,130)],fill=255); mask.save(mask_path)
    active={'sample_id':'DEMO_ONLY','image':str(image_path),'image_path':image_path,'question':'What color is the umbrella?','answers':['red'],'critical_concepts':['red umbrella'],'evidence_mask':str(mask_path),'candidate_answers':['red','blue','green','other'],'split':'demo'}
    IS_DEMO=True
print('Demo only:',IS_DEMO,'| sample:',active.get('sample_id'))

## 4. Inspect sample and parse question requirements

The deterministic parser is a transparent baseline, not a validated semantic parser. Use GQA scene graphs or manually reviewed annotations for critical entity/relation slots in the primary experiment.

In [ ]:
from semantic_circuits.evidence import parse_question_requirements
image=Image.open(active['image_path']).convert('RGB')
requirements=parse_question_requirements(active['question'])
print(json.dumps(requirements,indent=2))
display(image)

## 5. Generate nuisance variants

This starter creates mild brightness, contrast, and JPEG variants with unchanged dimensions. Geometry, color changes, background edits, and object removal need question-specific validity checks and are not applied automatically.

In [ ]:
from semantic_circuits.transforms import generate_photometric_variants, validity_gate
raw_variants=generate_photometric_variants(image,active['question'])
for variant in raw_variants: print(variant['transform']); display(variant['image'])
variant_dir=OUTPUT_DIR/'variants'/str(active.get('sample_id','sample')); variant_dir.mkdir(parents=True,exist_ok=True)
for variant in raw_variants: variant['image'].save(variant_dir/(variant['transform']+'.png'))

## 6. Human semantic-validity gate

Review every shown variant. Confirm the answer, question-critical objects, attributes, and relations are preserved. Edit `APPROVED_VARIANTS` only after that review. The gate defaults to rejecting every variant. The demo must never enter benchmark results.

In [ ]:
# Example after review: {'brightness_0.97','contrast_1.03','jpeg_quality_92'}
APPROVED_VARIANTS=set()
variant_records=[]
for variant in raw_variants:
    approved=variant['transform'] in APPROVED_VARIANTS
    variant_records.append({'transform':variant['transform'],'image':variant['image'],
        'answer_preserved':approved,'critical_evidence_preserved':approved,
        'relations_preserved':approved,'human_audited':approved})
[(v['transform'],validity_gate(v)) for v in variant_records]

## 7. Evidence mask: reviewed annotation or Grounding DINO + SAM2 proposal

The detector/segmenter only proposes a mask. Review its overlay before causal use. You can instead provide a manually reviewed mask in the manifest. Do not carry a mask across geometric transformations; re-ground and re-segment after any geometry change.

In [ ]:
RUN_GROUNDING=False  # set True to load Grounding DINO and SAM2 checkpoints
proposals=None
if RUN_GROUNDING:
    from semantic_circuits.grounding import GroundedEvidenceProposer
    proposer=GroundedEvidenceProposer(config['grounding_model'],config['sam2_model'],config.get('device','auto'),
        config.get('box_threshold',0.25),config.get('text_threshold',0.20),
        grounding_revision=config.get('grounding_revision'),sam2_revision=config.get('sam2_revision'))
    proposals=proposer(image,active.get('critical_concepts',[]))
    print('proposals:',len(proposals['candidates']))
    for item in proposals['candidates']:
        overlay=Image.fromarray((item['mask']*255).astype('uint8')).convert('RGBA'); overlay.putalpha(100)
        preview=image.convert('RGBA'); preview.alpha_composite(overlay); print(item['concept'],item['grounding_score'],item['mask_quality']); display(preview)



In [ ]:
RUN_CLIP=False  # optional semantic-ranking baseline for Grounding DINO proposals
clip_scorer=None; clip_ranked=[]
if RUN_CLIP and proposals and proposals['candidates']:
    from semantic_circuits.clip_proposals import ClipSemanticScorer,rank_grounded_candidates
    clip_scorer=ClipSemanticScorer(config.get('clip_model','openai/clip-vit-large-patch14'),config.get('device','auto'),config.get('clip_revision'))
    clip_ranked=rank_grounded_candidates(image,proposals['candidates'],clip_scorer)
    display(pd.DataFrame(clip_ranked).drop(columns=['box_xyxy_pixels'],errors='ignore'))
else:
    print('CLIP disabled; its score is an optional semantic proposal baseline, not causal evidence.')

In [ ]:
evidence_mask=None
if active.get('evidence_mask') and (IS_DEMO or active.get('evidence_mask_reviewed') is True):
    mask_path=resolve_path(active['evidence_mask'],DATA_ROOT,MANIFEST if MANIFEST.exists() else None)
    evidence_mask=np.asarray(Image.open(mask_path).convert('L'))>0
elif proposals and proposals['candidates']:
    # Explicitly approve reviewed proposal concepts; default is no accepted mask.
    APPROVED_EVIDENCE_CONCEPTS=set()
    chosen=[c for c in proposals['candidates'] if c['concept'] in APPROVED_EVIDENCE_CONCEPTS]
    if chosen: evidence_mask=np.logical_or.reduce([c['mask']>0 for c in chosen])
    else: print('No reviewed evidence mask selected. Set evidence_mask_reviewed=true or approve proposal labels.')
elif active.get('evidence_mask'):
    print('Mask exists but evidence_mask_reviewed is not true; inspect it and update the manifest after review.')
print('Reviewed/available mask:',evidence_mask is not None, 'shape:',None if evidence_mask is None else evidence_mask.shape)

In [ ]:
from semantic_circuits.controls import sample_translated_control_masks
control_candidates=[]; control_masks=[]
if evidence_mask is not None:
    try: control_candidates=sample_translated_control_masks(evidence_mask,n=3,seed=int(config.get('seed',17)))
    except RuntimeError as exc: print('Could not sample all controls:',exc)
    for idx,control in enumerate(control_candidates):
        overlay=Image.fromarray((control*255).astype('uint8')).convert('RGBA'); overlay.putalpha(100)
        preview=image.convert('RGBA'); preview.alpha_composite(overlay); print('control',idx); display(preview)
# Inspect overlays and set indices only for regions without other critical evidence.
APPROVED_CONTROL_IDS=set()
control_masks=[m for idx,m in enumerate(control_candidates) if idx in APPROVED_CONTROL_IDS]
print('Reviewed spatial controls:',len(control_masks))

## 8. Lightweight metric sanity check

These metric cells do not load a model. A pair of empty mediator sets scores zero CPS: shared absence is not a recovered stable circuit.

In [ ]:
from semantic_circuits.metrics import cps_hard, cps_weighted, ceca_scalar, ceca_from_output_distributions, scc_score
print('CPS example:',cps_hard([{'L1#head2','L2#head1'},{'L1#head2','L3#head4'}]))
print('CECA distribution example:',ceca_from_output_distributions({'red':.8,'blue':.2},{'red':.2,'blue':.8},{'red':.7,'blue':.3}))
print('SCC example:',scc_score(.82,.74))

## 9. Load hookable Qwen2.5-VL (optional, large checkpoint)

Qwen causal tracing is the expensive path. Set `RUN_CAUSAL=True` only after selecting a GPU and reviewing the model license/revision. `eager` attention is used so module hooks remain inspectable. Exact Transformers versions and checkpoint revisions must be saved with results.

In [ ]:
RUN_CAUSAL=False
model=None
if RUN_CAUSAL:
    from semantic_circuits.qwen_backend import Qwen25VL,QwenConfig
    model=Qwen25VL(QwenConfig(model_id=config['qwen_model'],revision=config.get('qwen_revision'),device=config.get('device','auto'),
                              dtype=config.get('dtype','auto'),max_new_tokens=config.get('max_new_tokens',48)))
    print('Loaded:',model.config.model_id,'| device:',model.device,'| hooked layers:',len(model.layers))
else:
    print('Qwen disabled; no checkpoint loaded.')

## 10. Baseline answer and fixed candidate-answer scoring

Candidate answers define a restricted answer distribution for paired intervention comparisons; it is not the model's full vocabulary distribution. Keep the same candidates, prompt, decoding, and model revision for all images in an instance.

In [ ]:
if model is not None:
    baseline_answer=model.answer(image,active['question'])
    candidates=active.get('candidate_answers') or list(dict.fromkeys(active.get('answers',[])))
    if baseline_answer not in candidates: candidates=[baseline_answer,*candidates]
    if len(candidates)<2: raise ValueError('Provide a fixed candidate_answers list with >=2 alternatives.')
    baseline_logprobs=model.answer_logprobs(image,active['question'],candidates)
    print('Baseline:',baseline_answer,'\nCandidate log probabilities:',baseline_logprobs)
else:
    baseline_answer=None; candidates=active.get('candidate_answers',[]); baseline_logprobs=None
    print('Enable RUN_CAUSAL and provide a fixed candidate-answer set to continue.')

## 11. External necessity and sufficiency interventions

Deletion preserves the full image canvas and replaces mask pixels with blur by default. Gray replacement is an ablation. Both can create artifacts, so use multiple controls and report intervention method. Evidence-only sufficiency retains the mask and neutralizes the rest of the image.

In [ ]:
if model is not None and evidence_mask is not None and len(candidates)>=2:
    from semantic_circuits.interventions import external_effects
    external=external_effects(image,active['question'],evidence_mask,candidates,model.answer_logprobs,
                              method=config.get('intervention_method','blur'),control_masks=control_masks)
    print(json.dumps(external,indent=2))
else:
    external=None; print('Skipped: model, reviewed evidence mask, or candidate answers unavailable.')

## 12. Recover pathway, compute CPS / CECA / SCC

This cell is guarded: it needs at least two human-approved variants, fixed candidates, and an evidence mask. It ranks attention-head input deltas, patches factual prompt-side head activations into evidence-deleted runs, and greedily recovers the target-answer log likelihood. The approximately minimal mediator set is not a proof of globally minimal causality.

In [ ]:
accepted_variants=[v for v in variant_records if validity_gate(v)[0]]
if model is not None and evidence_mask is not None and len(candidates)>=2 and len(accepted_variants)>=2:
    from semantic_circuits.runner import run_pathway_instance
    pathway=run_pathway_instance(model,image,active['question'],evidence_mask,candidates,accepted_variants,
        control_masks=control_masks,top_k=int(config.get('top_k',20)), recovery_threshold=float(config.get('recovery_threshold',0.8)),
        intervention_method=config.get('intervention_method','blur'))
    print(json.dumps(pathway,indent=2,ensure_ascii=False))
else:
    pathway=None
    print('Skipped. Need RUN_CAUSAL=True, an evidence mask, >=2 fixed candidates, and >=2 approved variants.')

## 13. Save sample result and reproducibility metadata

In [ ]:
if pathway is not None:
    pathway['sample_id']=active.get('sample_id'); pathway['split']=active.get('split')
    pathway['question_requirements']=requirements
    pathway['reproducibility']={'python':platform.python_version(),'platform':platform.platform(),
        'qwen_model':config['qwen_model'],'qwen_revision':config.get('qwen_revision'),
        'grounding_model':config['grounding_model'],'grounding_revision':config.get('grounding_revision'),
        'sam2_model':config['sam2_model'],'sam2_revision':config.get('sam2_revision'),
        'dtype':config.get('dtype'),'top_k':config.get('top_k'),'intervention_method':config.get('intervention_method'),
        'seed':config.get('seed'),'transformers_version':package_metadata.version('transformers'),
        'torch_version':package_metadata.version('torch')}
    output_path=OUTPUT_DIR/f"{active.get('sample_id','sample')}_pathway.json"
    output_path.write_text(json.dumps(pathway,indent=2,ensure_ascii=False),encoding='utf-8')
    print('Saved:',output_path)
else:
    print('No causal result exported; save only after the guarded causal experiment completes.')

## 14. Research interpretation and failure checks

Inspect before aggregating: low target deletion effect may mean the detector mask missed evidence, or the model used redundant evidence; poor patch recovery may indicate distributed mediation or invalid interventions; low CPS can reflect nuisance sensitivity, token alignment changes, or an unstable model circuit. Compare against behavioral confidence, CLIP/grounding baselines, external-only necessity/sufficiency, and internal-only attribution. The architecture claims do not follow from one sample or from demo data.

## 15. Run the reviewed manifest end to end

Set `RUN_BATCH_EXPERIMENT=True` only after configuring the manifest, reviewing masks, and applying variant review decisions. The runner resumes by `sample_id` and appends JSONL results. Start with `BATCH_LIMIT=1`; use `None` for all remaining manifest rows.

In [ ]:
RUN_BATCH_EXPERIMENT=False
BATCH_LIMIT=1  # set to None to run all remaining samples
if RUN_BATCH_EXPERIMENT:
    if not MANIFEST.exists(): raise FileNotFoundError(f'Configured manifest not found: {MANIFEST}')
    from scripts.run_experiment import run_manifest
    result_file=run_manifest(config,limit=BATCH_LIMIT,resume=True)
    print('Results: ',result_file)
else:
    print('Batch run disabled. Set RUN_BATCH_EXPERIMENT=True after completing the reviewed-manifest preparation steps.')